# 02C — Diagnose ViTPose on Five Biking Frames

This diagnostic notebook isolates the crash seen during full-video extraction.

**What this notebook does**
- Loads the existing ViTPose checkpoint on CPU.
- Loads the existing YOLO detector on CPU.
- Reads one Biking video and processes only 5 sampled frames (original frame indices 0, 4, 8, 12, 16).
- Runs person detection and ViTPose pose estimation on at most one person per sampled frame.
- Saves individual annotated JPEGs and a compact `.npz` pose file.

**What it deliberately does NOT do**
- It does not use the tracker.
- It does not write an MP4.
- It does not train either model.
- It does not process the full video.

Run cells one at a time with Shift+Enter. If the kernel dies, note the last printed message.

## Cell 1 — Load libraries and models

Run this cell once. CPU is used intentionally to isolate the issue from CUDA.

In [1]:
from pathlib import Path
import sys
import gc
import time
import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

PROJECT_DIR = Path(r"C:\Users\admin\ViTPose_Action_Recognition")
sys.path.insert(0, str(PROJECT_DIR))

from src.vitpose_model import (
    ViTPose, get_affine_transform, decode_heatmaps, draw_skeleton_on,
    IMG_H, IMG_W, IMG_MEAN, IMG_STD
)

DEVICE = torch.device("cpu")
torch.set_num_threads(2)

CHECKPOINT_PATH = PROJECT_DIR / "checkpoints" / "epoch_209.pth"
YOLO_PATH = PROJECT_DIR / "notebooks" / "yolo11n.pt"
MANIFEST_PATH = PROJECT_DIR / "data" / "processed" / "manifest.csv"

assert CHECKPOINT_PATH.is_file(), f"Checkpoint not found: {CHECKPOINT_PATH}"
assert YOLO_PATH.is_file(), f"YOLO weights not found: {YOLO_PATH}"
assert MANIFEST_PATH.is_file(), f"Manifest not found: {MANIFEST_PATH}"

print("Loading ViTPose checkpoint on CPU...", flush=True)
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
model = ViTPose()
model.load_state_dict(checkpoint["model_state"], strict=True)
model.eval()
print("ViTPose loaded; checkpoint epoch:", checkpoint.get("epoch", "unknown"), flush=True)
del checkpoint
gc.collect()

print("Loading YOLO on CPU...", flush=True)
detector = YOLO(str(YOLO_PATH))
print("YOLO loaded.", flush=True)
print("Device:", DEVICE, flush=True)

Loading ViTPose checkpoint on CPU...
ViTPose loaded; checkpoint epoch: 209
Loading YOLO on CPU...
YOLO loaded.
Device: cpu


## Cell 2 — Select one Biking video and create output folder

Check that the printed path ends in a Biking video and that `Video exists: True`.

In [2]:
manifest = pd.read_csv(MANIFEST_PATH)
biking = manifest[manifest["label"].astype(str).str.casefold().eq("biking")].reset_index(drop=True)
assert len(biking) > 0, "No Biking videos found in manifest."

VIDEO_PATH = Path(str(biking.loc[0, "video_path"]))
VIDEO_LABEL = str(biking.loc[0, "label"])
assert VIDEO_PATH.is_file(), f"Video not found: {VIDEO_PATH}"

DEBUG_DIR = PROJECT_DIR / "outputs" / "pose_videos" / "debug_biking_five_frames"
DEBUG_DIR.mkdir(parents=True, exist_ok=True)
NPZ_PATH = PROJECT_DIR / "data" / "processed" / "keypoints_single_tracked" / "biking_debug_five_frames.npz"
NPZ_PATH.parent.mkdir(parents=True, exist_ok=True)

FRAME_STEP = 4
MAX_SAMPLED_FRAMES = 5
DETECTION_CONF = 0.25
POSE_THRESH = 0.10

print("Label:", VIDEO_LABEL)
print("Video:", VIDEO_PATH)
print("Video exists:", VIDEO_PATH.is_file())
print("Output JPEG folder:", DEBUG_DIR)
print("Only sampled frames 0, 4, 8, 12, 16 will be tested.")

Label: Biking
Video: C:\Users\admin\ViTPose_Action_Recognition\data\raw\ucf_sports\Biking\v_Biking_g01_c01.avi
Video exists: True
Output JPEG folder: C:\Users\admin\ViTPose_Action_Recognition\outputs\pose_videos\debug_biking_five_frames
Only sampled frames 0, 4, 8, 12, 16 will be tested.


## Cell 3 — Define pose function

This uses the same crop and normalization functions expected by the trained model.

In [3]:
def pose_for_box_cpu(frame_rgb, bbox):
    x1, y1, x2, y2 = map(float, bbox)
    h, w = frame_rgb.shape[:2]
    x1 = np.clip(x1, 0, w - 1)
    x2 = np.clip(x2, 0, w)
    y1 = np.clip(y1, 0, h - 1)
    y2 = np.clip(y2, 0, h)
    bw, bh = x2 - x1, y2 - y1
    if bw < 2 or bh < 2:
        return None

    center = np.array([(x1 + x2) / 2, (y1 + y2) / 2], dtype=np.float32)
    scale = max(bw, bh * IMG_W / IMG_H) * 1.25
    trans = get_affine_transform(center, scale, 0.0, (IMG_W, IMG_H))
    crop = cv2.warpAffine(frame_rgb, trans, (IMG_W, IMG_H))
    crop = crop.astype(np.float32) / 255.0
    crop = (crop - IMG_MEAN) / IMG_STD
    tensor = torch.from_numpy(crop.transpose(2, 0, 1).copy()).float().unsqueeze(0)

    with torch.inference_mode():
        heatmaps = model(tensor)[0].float().cpu().numpy()

    keypoints = decode_heatmaps(heatmaps, center, scale)
    del tensor, heatmaps, crop
    return keypoints.astype(np.float32)

print("CPU pose function is ready.", flush=True)

CPU pose function is ready.


## Cell 4 — Process exactly five sampled frames

This cell intentionally skips tracking and MP4 writing. It saves one JPEG per sampled frame and the keypoints to one `.npz` file.

In [4]:
cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened(), f"Cannot open video: {VIDEO_PATH}"

observations = []
frame_index = 0
sampled_count = 0
started = time.time()

try:
    while sampled_count < MAX_SAMPLED_FRAMES:
        ok, frame_bgr = cap.read()
        if not ok:
            print("Reached end of video.", flush=True)
            break

        if frame_index % FRAME_STEP == 0:
            print(f"Sample {sampled_count + 1}/{MAX_SAMPLED_FRAMES}; source frame {frame_index}: reading", flush=True)
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

            print("  YOLO detection...", flush=True)
            result = detector.predict(
                frame_bgr, classes=[0], conf=DETECTION_CONF,
                device="cpu", verbose=False
            )[0]

            if result.boxes is None or len(result.boxes) == 0:
                print("  No person detected in this sampled frame.", flush=True)
                annotated_rgb = frame_rgb.copy()
            else:
                boxes = result.boxes.xyxy.cpu().numpy()
                areas = np.maximum(0, boxes[:, 2] - boxes[:, 0]) * np.maximum(0, boxes[:, 3] - boxes[:, 1])
                bbox = boxes[int(np.argmax(areas))].astype(np.float32)

                print("  ViTPose inference...", flush=True)
                keypoints = pose_for_box_cpu(frame_rgb, bbox)
                print("  Pose shape:", keypoints.shape,
                      "| valid xy joints:", int(np.isfinite(keypoints[:, :2]).all(axis=1).sum()), "/ 17",
                      flush=True)

                annotated_rgb = draw_skeleton_on(frame_rgb.copy(), keypoints, thresh=POSE_THRESH)
                x1, y1, x2, y2 = map(int, bbox)
                cv2.rectangle(annotated_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)

                observations.append({
                    "frame_idx": int(frame_index),
                    "keypoints": keypoints,
                    "box": bbox
                })

            jpeg_path = DEBUG_DIR / f"frame_{frame_index:06d}.jpg"
            saved = cv2.imwrite(str(jpeg_path), cv2.cvtColor(annotated_rgb, cv2.COLOR_RGB2BGR))
            assert saved, f"Could not save JPEG: {jpeg_path}"
            print("  Saved:", jpeg_path.name, flush=True)

            sampled_count += 1
            del frame_rgb, result, annotated_rgb
            gc.collect()

        frame_index += 1
finally:
    cap.release()

print("\nFive-frame diagnostic finished.", flush=True)
print("Source frames read:", frame_index, flush=True)
print("Sampled frames attempted:", sampled_count, flush=True)
print("Frames with pose observations:", len(observations), flush=True)
print("Elapsed seconds:", round(time.time() - started, 1), flush=True)

if observations:
    pose_data = {
        "keypoints": np.stack([o["keypoints"] for o in observations]).astype(np.float32),
        "boxes": np.stack([o["box"] for o in observations]).astype(np.float32),
        "frame_indices": np.asarray([o["frame_idx"] for o in observations], dtype=np.int32),
        "label": np.asarray(VIDEO_LABEL),
        "video_path": np.asarray(str(VIDEO_PATH)),
        "frame_step": np.asarray(FRAME_STEP, dtype=np.int32),
    }
    temp_path = NPZ_PATH.with_name(NPZ_PATH.stem + ".tmp.npz")
    np.savez_compressed(temp_path, **pose_data)
    temp_path.replace(NPZ_PATH)
    print("Saved keypoints:", NPZ_PATH, flush=True)
else:
    print("No poses were saved; inspect YOLO detection results.", flush=True)

Sample 1/5; source frame 0: reading
  YOLO detection...
  ViTPose inference...
  Pose shape: (17, 3) | valid xy joints: 17 / 17
  Saved: frame_000000.jpg
Sample 2/5; source frame 4: reading
  YOLO detection...
  ViTPose inference...
  Pose shape: (17, 3) | valid xy joints: 17 / 17
  Saved: frame_000004.jpg
Sample 3/5; source frame 8: reading
  YOLO detection...
  ViTPose inference...
  Pose shape: (17, 3) | valid xy joints: 17 / 17
  Saved: frame_000008.jpg
Sample 4/5; source frame 12: reading
  YOLO detection...
  ViTPose inference...
  Pose shape: (17, 3) | valid xy joints: 17 / 17
  Saved: frame_000012.jpg
Sample 5/5; source frame 16: reading
  YOLO detection...
  ViTPose inference...
  Pose shape: (17, 3) | valid xy joints: 17 / 17
  Saved: frame_000016.jpg

Five-frame diagnostic finished.
Source frames read: 17
Sampled frames attempted: 5
Frames with pose observations: 5
Elapsed seconds: 4.1
Saved keypoints: C:\Users\admin\ViTPose_Action_Recognition\data\processed\keypoints_single

## Cell 5 — Verify outputs

If all cells completed, this prints the saved JPEGs and pose file shape. Open the JPEGs from the folder with Jupyter's file browser.

In [5]:
jpeg_files = sorted(DEBUG_DIR.glob("*.jpg"))
print("Debug JPEGs:", len(jpeg_files))
for p in jpeg_files:
    print(p.name, round(p.stat().st_size / 1024, 1), "KB")

print("NPZ exists:", NPZ_PATH.is_file())
if NPZ_PATH.is_file():
    with np.load(NPZ_PATH, allow_pickle=False) as data:
        print("Saved keypoints shape:", data["keypoints"].shape, "(T, 17, 3)")
        print("Frame indices:", data["frame_indices"].tolist())
        print("Label:", data["label"].item())

Debug JPEGs: 5
frame_000000.jpg 17.2 KB
frame_000004.jpg 22.2 KB
frame_000008.jpg 23.5 KB
frame_000012.jpg 17.5 KB
frame_000016.jpg 22.8 KB
NPZ exists: True
Saved keypoints shape: (5, 17, 3) (T, 17, 3)
Frame indices: [0, 4, 8, 12, 16]
Label: Biking


## How to interpret this test

- **If this notebook completes:** video reading, YOLO, ViTPose, and JPEG/NPZ output work for a short run. The previous failure is likely in a component excluded here (tracker/MP4 writing), or cumulative work in the full loop. We will add components back one at a time.
- **If the kernel dies:** note the last printed step (`YOLO detection` vs `ViTPose inference` vs `Saved`). That pinpoints the failing operation.
- This is only a diagnostic subset; it is not yet the full video output or action recognition.